# 🤟 Ishara — Complete Google Colab GPU Training Notebook

This notebook automatically fetches the **INCLUDE Indian Sign Language dataset from Zenodo Record 4010759**, extracts 225-dim MediaPipe keypoints, and trains the **Bi-LSTM with Attention** model on GPU for real-world live translation.

---  
### 📋 Instructions for Google Colab:
1. Go to **Runtime → Change runtime type** and select **T4 GPU**.
2. Upload your updated `Ishara.zip` project file.
3. Run all cells sequentially from Step 1 to Step 8.
4. After Step 7 finishes, download `checkpoints/best_model.pth` and place it in your local `checkpoints/` folder.


## Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ WARNING: GPU not detected. Go to Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Install Project Dependencies

In [ ]:
!pip install -q mediapipe opencv-python-headless torch torchvision pandas pyyaml tqdm scikit-learn matplotlib seaborn google-genai google-generativeai

## Step 3: Unpack Project Workspace & Set Paths

In [ ]:
import os, sys
from google.colab import files

# Check if uploaded Ishara.zip exists in /content/
if os.path.exists('/content/Ishara.zip'):
    !unzip -o -q /content/Ishara.zip -d /content/Ishara/
elif not os.path.exists('/content/Ishara/src') and not os.path.exists('src'):
    print("📦 Please upload 'Ishara.zip':")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            !unzip -o -q "{fn}" -d /content/Ishara/

if os.path.exists('/content/Ishara'):
    %cd /content/Ishara

sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())
print("Vocabulary file exists:", os.path.exists('data/vocabulary.json'))
print("Files present:", os.listdir('.'))

## Step 4: Download INCLUDE ISL Dataset from Zenodo Record 4010759

In [ ]:
import os, sys
if os.path.exists('/content/Ishara'):
    %cd /content/Ishara
sys.path.insert(0, os.getcwd())

from src.data.download_include import download_include_dataset, filter_vocabulary_videos

# Download raw dataset from Zenodo API
print("Downloading INCLUDE Sign Language dataset...")
download_include_dataset(output_dir="data/raw/include", subset_only=False)

# Filter dataset matching active 65-word hospital vocabulary
df_filtered = filter_vocabulary_videos(include_dir="data/raw/include", vocab_file="data/vocabulary.json", output_csv="data/train_split.csv")
print(f"Indexed {len(df_filtered)} videos matching active vocabulary.")

## Step 5: Extract 225-dim MediaPipe Landmark Keypoints

In [ ]:
import sys, os
if os.path.exists('/content/Ishara'):
    %cd /content/Ishara
sys.path.insert(0, os.getcwd())

from src.data.extract_keypoints import batch_extract_dataset

print("Extracting MediaPipe Pose + Hand keypoints for train, val, and test splits...")
if os.path.exists('data/train_split.csv'):
    batch_extract_dataset('data/train_split.csv', 'data/processed/train')
if os.path.exists('data/val_split.csv'):
    batch_extract_dataset('data/val_split.csv', 'data/processed/val')
if os.path.exists('data/test_split.csv'):
    batch_extract_dataset('data/test_split.csv', 'data/processed/test')

## Step 6: Train Bi-LSTM Model on GPU

In [ ]:
import sys, os
if os.path.exists('/content/Ishara'):
    %cd /content/Ishara
sys.path.insert(0, os.getcwd())

from src.model.train import run_training

# Start PyTorch GPU model training
print("Launching GPU training...")
run_training('config.yaml')

## Step 7: Evaluate Model Performance & Confusion Matrix

In [ ]:
import sys, os
if os.path.exists('/content/Ishara'):
    %cd /content/Ishara
sys.path.insert(0, os.getcwd())

from src.model.eval_test import run_test_evaluation
run_test_evaluation('config.yaml')

## Step 8: Download Trained Checkpoint (`best_model.pth`)

In [ ]:
from google.colab import files
import os

checkpoint_file = 'checkpoints/best_model.pth'
if os.path.exists(checkpoint_file):
    print("📥 Triggering browser download for best_model.pth...")
    files.download(checkpoint_file)
else:
    print(f"⚠️ Checkpoint '{checkpoint_file}' not found.")